# R3: what happens to new pump.fun coins in their first hour

Research step R3 (spec 15.3): a first look at the dataset before any model. The question
of the project: using only a coin's first 30 s to 5 min of on-chain activity, how well can
we predict whether it dies, rugs, graduates or doubles?

This notebook only describes the data. It reads `dataset/dataset.parquet` in the data
folder (made by `agent dataset`) and runs top to bottom; rebuild the dataset and run it
again to update every number. The findings, with numbers, are in `docs/PROGRESS.md`.

- **Held-out days.** Only rows before the validation and test days of the evaluation plan
  (spec 15.4, `config/evaluation.yaml`) are used, so later tests stay untouched. While the
  data has too few whole days for that split, nothing is held out.
- **Intervals** are 95%, from resampling whole launch hours: coins launched in the same
  hour share market conditions, so they are not independent.
- **Groups:** *ordinary* (paired with SOL), *mayhem* (paired with SOL, traded by pump.fun's
  own bot) and *other quote* (priced in another coin: no price labels).
- Columns starting with `label_` look into the future. They are outcomes here, never inputs.

In [ ]:
import json
from datetime import UTC, datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.ticker import PercentFormatter
from sklearn.metrics import average_precision_score, roc_auc_score

from agent import clock
from agent.config import load_config
from agent.models.evaluate import hour_interval, load_plan, split_days

cfg = load_config()
folder = cfg.data_dir / "dataset"
manifest = json.loads((folder / "manifest.json").read_text(encoding="utf-8"))
plan, plan_sha = load_plan(cfg.root / "config" / "evaluation.yaml")
data = pd.read_parquet(folder / manifest["file"], dtype_backend="numpy_nullable")
split = split_days([tuple(s) for s in manifest["backfill_spans"]], plan.split)
CI = {"level": plan.uncertainty.level, "resamples": plan.uncertainty.resamples,
      "seed": plan.uncertainty.seed}
OFFSETS = manifest["offsets_s"]

# Chart style: light surface, recessive hairline grid, one fixed color per group.
SURFACE, INK, INK2, MUTED, GRID, BASELINE = (
    "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7")
GROUPS = ["ordinary", "mayhem", "other quote"]
COLOR = {"ordinary": "#2a78d6", "mayhem": "#eb6834", "other quote": "#1baf7a"}
REST = "#c3c2b7"  # the "everything else" gray when one series is highlighted
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "figure.dpi": 110, "font.family": ["Segoe UI", "DejaVu Sans"], "font.size": 9.5,
    "text.color": INK, "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "axes.titlesize": 10.5, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.edgecolor": BASELINE, "axes.linewidth": 0.75,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.75, "grid.linestyle": "-",
    "axes.axisbelow": True, "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "lines.linewidth": 1.5, "lines.solid_capstyle": "round", "lines.solid_joinstyle": "round",
    "legend.frameon": False, "legend.labelcolor": INK2,
})


def num(series):
    # A column as floats, with unknown values as NaN.
    return series.astype("Float64").to_numpy(dtype=float, na_value=np.nan)


def ci(frame, column):
    # Mean of a 0/1 (or numeric) column with its 95% interval over launch hours.
    return hour_interval(num(frame[column]), frame["hour_ts"].to_numpy(), **CI)


def pct(r, digits=1):
    return "-" if np.isnan(r) else f"{r:.{digits}%}"


def with_ci(values, digits=1):
    r, lo, hi = values
    return "-" if np.isnan(r) else f"{r:.{digits}%} ({lo:.{digits}%} to {hi:.{digits}%})"


def median(values):
    values = values[~np.isnan(values)]
    return float(np.median(values)) if len(values) else np.nan


def as_pct(ax):
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=None))


def hour_name(h):
    return f"{h % 12 or 12} {'AM' if h < 12 else 'PM'}"


def end_label(ax, x, y, text):
    # A direct label at the end of a line, in text ink (never the series color).
    ax.annotate(text, (x, y), xytext=(6, 0), textcoords="offset points", va="center",
                color=INK2, fontsize=9)


group = np.where(~data["is_sol_paired"].to_numpy(bool), "other quote",
                 np.where(data["is_mayhem"].to_numpy(bool), "mayhem", "ordinary"))
data["group"] = pd.Categorical(group, categories=GROUPS)
data["role"] = [split.role(int(t)) for t in data["created_ts"]]
usable = ~data["has_gap"].to_numpy(bool) & data["labels_complete"].to_numpy(bool)
explore = data[usable & (data["role"] == "train").to_numpy()].copy()
coins = explore[explore["offset_s"] == OFFSETS[0]].copy()  # one row per coin
# Graduating *after* the decision time, for coins still on the curve then (section 2b).
explore["on_curve"] = ~explore["completed"].to_numpy(bool)
explore["graduates_later"] = (explore["label_graduated_1h"].fillna(False).to_numpy(bool)
                              & explore["on_curve"]).astype(float)

## 1. The data

In [ ]:
spans = manifest["backfill_spans"]
days = ", ".join(datetime.fromtimestamp(d, UTC).strftime("%b %d") for d in split.whole_days)
print(f"Backfill ({clock.zone()}): {clock.stamp(spans[0][0], seconds=False)} to "
      f"{clock.stamp(spans[-1][1], seconds=False)}; epochs {manifest['epochs']}")
print(f"Dataset: {manifest['rows']:,} rows ({manifest['coins']:,} coins x {len(OFFSETS)} "
      f"cutoffs), file hash {manifest['file_sha256'][:12]}, made by code "
      f"{(manifest['code']['git_commit'] or '?')[:7]} (source {manifest['code']['source_sha']})")
print(f"Whole UTC days: {len(split.whole_days)} ({days or 'none'}). Evaluation plan "
      f"{plan_sha}: ", end="")
if split.explore_until is None:
    need = plan.split.min_test_days + plan.split.validation_days + plan.split.min_train_days
    print(f"too few whole days for the held-out split (it needs {need}), so nothing is "
          "held out yet.")
else:
    print(f"rows launched from {clock.stamp(split.explore_until, seconds=False)} on are held "
          f"out ({len(split.validation)} validation and {len(split.test)} test days).")
print(f"Used here: {len(coins):,} coins with every label known and no gap, "
      f"{len(explore):,} rows.")

summary = coins.groupby("group", observed=False).size().rename("coins").to_frame()
summary["share"] = (summary["coins"] / summary["coins"].sum()).map(lambda v: f"{v:.0%}")
summary["median trades in the hour"] = coins.groupby("group", observed=False)[
    "label_trades_1h"].median().astype(int)
display(summary)

In [ ]:
per_hour = (coins.groupby(["hour_ts", "group"], observed=False).size()
            .unstack("group").reindex(columns=GROUPS).fillna(0))
per_hour = per_hour.iloc[1:-1]  # the first and last hours are only partly in the data
fig, ax = plt.subplots(figsize=(8.5, 3.4), layout="constrained")
for name in GROUPS:
    ax.plot(per_hour.index, per_hour[name], color=COLOR[name], label=name)
ax.set_title("Coins launched per hour (whole hours only)")
ax.set_ylabel("coins per hour")
ax.set_ylim(bottom=0)
ticks = per_hour.index[::3]
ax.set_xticks(ticks, [f"{clock.stamp(t, seconds=False).rsplit(':', 1)[0]} "
                      f"{clock.stamp(t, seconds=False).split()[-1]}" for t in ticks],
              rotation=0, fontsize=8)
ax.legend(loc="upper left", ncols=3)
plt.show()
display(per_hour.astype(int).rename(index=lambda t: clock.stamp(t, seconds=False)))

## 2. The first hour: how often coins die, rug, look bundled or graduate

Labels (spec 7, `config/labels.yaml`): **dead** = no trade after 30 min; **rugged** = the
creator sold at least half of what they bought and the price fell 80% from its peak
(SOL coins only); **bundled** is a stand-in: wallets other than the creator that bought in
the first second hold at least 20% of supply at T+60 s; **bad** = any of the three;
**graduated** = the bonding curve completed within the hour.

In [ ]:
LABELS = {"dead": "label_is_dead", "dead, not counting the bot": "label_is_dead_without_agent",
          "rugged": "label_is_rug", "bundled (stand-in)": "label_is_bundled_proxy",
          "bad": "label_bad", "graduated": "label_graduated_1h"}
rates = {}
for name in GROUPS:
    frame = coins[coins["group"] == name]
    for label, column in LABELS.items():
        if column in ("label_is_rug", "label_bad") and name == "other quote":
            rates[(label, name)] = (np.nan, np.nan, np.nan)  # no price labels
        else:
            rates[(label, name)] = ci(frame, column)
table = pd.DataFrame({name: {label: with_ci(rates[(label, name)]) for label in LABELS}
                      for name in GROUPS})
display(table)

shown = ["dead", "rugged", "bundled (stand-in)", "graduated"]
fig, axes = plt.subplots(1, len(shown), figsize=(10, 2.6), layout="constrained", sharey=True)
for ax, label in zip(axes, shown, strict=True):
    for i, name in enumerate(GROUPS):
        r, lo, hi = rates[(label, name)]
        if np.isnan(r):
            continue
        ax.plot([lo, hi], [i, i], color=COLOR[name], linewidth=1.5)
        ax.plot(r, i, "o", color=COLOR[name], markersize=7, markeredgecolor=SURFACE,
                markeredgewidth=1.5, label=name)
        ax.annotate(pct(r), (hi, i), xytext=(5, 0), textcoords="offset points",
                    va="center", color=INK2, fontsize=8.5)
    ax.set_title(label)
    ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.set_xlim(0, max(rates[(label, n)][2] for n in GROUPS
                       if not np.isnan(rates[(label, n)][0])) * 1.35)
    ax.grid(axis="y", visible=False)
axes[0].set_yticks(range(len(GROUPS)), GROUPS)
axes[0].invert_yaxis()
fig.suptitle("Share of coins, first hour (95% intervals over launch hours; each panel "
             "has its own scale)", x=0.01, ha="left", color=INK2, fontsize=9)
plt.show()

### 2b. Many coins graduate before any decision time

Some coins complete their bonding curve in the very second they launch: a handful of
wallets buy the whole curve in the launch block. At T+30 s their signals already show a
complete curve, so "will it graduate within the hour?" is already answered for them.
The question a model can usefully answer is the next table's last column: of the coins
**still on the curve** at the decision time, how many graduate later in the hour?

In [ ]:
table = {}
for name in GROUPS:
    for offset in OFFSETS:
        frame = explore[(explore["group"] == name) & (explore["offset_s"] == offset)]
        grad = frame["label_graduated_1h"].fillna(False).to_numpy(bool)
        done = ~frame["on_curve"].to_numpy(bool)
        on_curve = frame[frame["on_curve"].to_numpy(bool)]
        table[(name, f"T+{offset} s")] = {
            "graduated in the hour": int(grad.sum()),
            "of those, already complete": pct((grad & done).sum() / max(grad.sum(), 1), 0),
            "still on the curve": len(on_curve),
            "graduate later (share of those)": with_ci(ci(on_curve, "graduates_later"), 2),
        }
display(pd.DataFrame(table).T)
instant = coins[coins["label_graduated_after_s"].fillna(10**9).to_numpy(float) <= 1]
print(f"Graduated within 1 s of launch: {len(instant):,} coins "
      f"({(instant['group'] == 'ordinary').sum():,} ordinary); median wallets buying in the "
      f"launch slot: {median(num(instant['launch_slot_buyers'])):.0f}.")

## 3. Doubling before halving

`hit_2x`: a simulated 1 SOL buy 5 s after the decision time (with its price impact, before
fees) saw a later price of 2x before one of 0.5x, within the hour. SOL coins only. By
default the mayhem bot's own trade prices are left out; the last row includes them.

In [ ]:
rows = {}
for name in ("ordinary", "mayhem"):
    frame = explore[explore["group"] == name]
    rows[name] = {o: ci(frame[frame["offset_s"] == o], "label_hit_2x") for o in OFFSETS}
mayhem = explore[explore["group"] == "mayhem"]
rows["mayhem, with the bot's prices"] = {
    o: ci(mayhem[mayhem["offset_s"] == o], "label_hit_2x_with_agent_prices") for o in OFFSETS}
display(pd.DataFrame({name: {f"T+{o} s": with_ci(v) for o, v in by.items()}
                      for name, by in rows.items()}).T)

fig, ax = plt.subplots(figsize=(6.5, 3.3), layout="constrained")
x = np.arange(len(OFFSETS))
for name in ("ordinary", "mayhem"):
    r = np.array([rows[name][o] for o in OFFSETS])
    ax.fill_between(x, r[:, 1], r[:, 2], color=COLOR[name], alpha=0.12, linewidth=0)
    ax.plot(x, r[:, 0], color=COLOR[name], marker="o", markersize=6,
            markeredgecolor=SURFACE, markeredgewidth=1.5, label=name)
    end_label(ax, x[-1], r[-1, 0], f"{name} {pct(r[-1, 0])}")
ax.set_xticks(x, [f"T+{o} s" for o in OFFSETS])
ax.set_xlim(-0.2, len(OFFSETS) - 0.4)
as_pct(ax)
ax.set_ylim(bottom=0)
ax.set_title("Reached 2x before 0.5x after a 1 SOL buy at the decision time + 5 s")
ax.set_xlabel("decision time")
ax.legend(loc="upper right")
plt.show()

## 4. How long coins keep trading

The share of coins that still had a trade after *t* minutes (the mayhem bot's trades
count), and how fast the coins that graduated got there.

In [ ]:
minutes = np.arange(0, 60)
life = {name: np.nan_to_num(num(coins.loc[coins["group"] == name, "label_last_trade_after_s"]))
        for name in GROUPS}
fig, ax = plt.subplots(figsize=(8, 3.4), layout="constrained")
for name in GROUPS:
    share = (life[name][:, None] > minutes * 60).mean(axis=0)
    ax.plot(minutes, share, color=COLOR[name], label=name)
as_pct(ax)
ax.set_ylim(0, 1.02)
ax.set_xlim(0, 60)
ax.set_xlabel("minutes after launch")
ax.set_title("Share of coins still trading")
ax.legend(loc="upper right")
plt.show()

checkpoints = [1, 2, 5, 15, 30, 45]
table = pd.DataFrame({name: {**{f"after {m} min": pct((life[name] > m * 60).mean(), 0)
                                for m in checkpoints},
                             "median last trade": f"{np.median(life[name]):.0f} s"}
                      for name in GROUPS})
display(table)

graduated = coins[coins["label_graduated_1h"].fillna(False).to_numpy(bool)]
quantiles = (graduated.groupby("group", observed=False)["label_graduated_after_s"]
             .quantile([0.25, 0.5, 0.75]).unstack() / 60)
quantiles.columns = ["25% (min)", "median (min)", "75% (min)"]
quantiles.insert(0, "graduated coins", graduated.groupby("group", observed=False).size())
display(quantiles.round(1))

## 5. Hour of day

Launch time on the PC's clock (Pacific time). **Caution:** with about one day of data,
each hour of the day appears once or twice, so these mix the time of day with whatever
happened in that particular hour (one unusual hour can stand out on its own). No
intervals: there is nothing to resample within an hour of the day yet.

In [ ]:
coins["local_hour"] = [datetime.fromtimestamp(int(t)).hour for t in coins["created_ts"]]
seen = coins.groupby("local_hour")["hour_ts"].nunique()  # how many times each hour appears
launches = (coins.groupby(["local_hour", "group"], observed=False).size().unstack("group")
            .reindex(columns=GROUPS).div(seen, axis=0))
ordinary = coins[coins["group"] == "ordinary"]
at60 = explore[(explore["group"] == "ordinary") & (explore["offset_s"] == 60)]
by_hour = pd.DataFrame({
    "bad": ordinary.groupby("local_hour")["label_bad"].apply(lambda s: np.nanmean(num(s))),
    "2x from T+60 s": at60.groupby(at60["created_ts"].map(
        lambda t: datetime.fromtimestamp(int(t)).hour))["label_hit_2x"].apply(
        lambda s: np.nanmean(num(s))),
})

hours = launches.index.to_numpy()
fig, axes = plt.subplots(1, 3, figsize=(11, 3), layout="constrained")
for name in GROUPS:
    axes[0].plot(hours, launches[name], color=COLOR[name], label=name)
axes[0].set_title("Launches per hour")
axes[0].legend(loc="upper left", fontsize=8)
for ax, column in zip(axes[1:], by_hour.columns, strict=True):
    ax.plot(hours, by_hour[column], color=COLOR["ordinary"], marker="o", markersize=4)
    ax.set_title(f"Ordinary coins: {column}")
    as_pct(ax)
    ax.set_ylim(bottom=0)
for ax in axes:
    ax.set_xticks(range(0, 24, 4), [hour_name(h) for h in range(0, 24, 4)])
    ax.set_xlabel(f"launch hour ({clock.zone()})")
axes[0].set_ylim(bottom=0)
plt.show()

table = launches.round(0).astype(int)
table["ordinary: bad"] = by_hour["bad"].map(pct)
table["ordinary: 2x from T+60 s"] = by_hour["2x from T+60 s"].map(pct)
table.index = [hour_name(h) for h in table.index]
table.index.name = f"launch hour ({clock.zone()})"
display(table)

## 6. Mayhem coins against ordinary ones

Medians at T+60 s (signals) and over the first hour (labels).

In [ ]:
at = explore[explore["offset_s"] == 60]
signals = ["trades", "unique_buyers", "holders", "buy_sol", "top10_hold_pct",
           "dev_buy_pct", "curve_progress_pct", "mayhem_agent_buy_share"]
outcomes = ["label_trades_1h", "label_agent_trades_1h", "label_last_trade_after_s",
            "label_peak_multiple", "label_max_drawdown", "label_creator_sold_share"]
table = pd.DataFrame({
    name: {**{f"T+60 s: {s}": median(num(at.loc[at["group"] == name, s])) for s in signals},
           **{f"hour: {c[6:]}": median(num(coins.loc[coins["group"] == name, c]))
              for c in outcomes}}
    for name in GROUPS})
display(table.round(3))

## 7. Creator history

How many coins each creator launched in the data, and whether a creator's earlier
launches go with how a coin does. `dev_prior_launches` only counts launches inside the
backfill, so coins early in the data look like first launches; the second table keeps
coins with at least 12 h of backfill before them.

In [ ]:
per_creator = coins.groupby("creator").size()
buckets = pd.cut(per_creator, [0, 1, 3, 10, np.inf], labels=["1", "2-3", "4-10", "11+"])
table = pd.DataFrame({"creators": per_creator.groupby(buckets, observed=False).size(),
                      "share of coins": per_creator.groupby(buckets, observed=False).sum()
                      / per_creator.sum()})
table["share of coins"] = table["share of coins"].map(lambda v: f"{v:.0%}")
table.index.name = "coins per creator"
display(table)
print(f"{per_creator.size:,} creators; the busiest launched {per_creator.max():,} coins "
      "(addresses are not shown).")

BINS, NAMES = [-1, 0, 1, 4, 19, np.inf], ["0", "1", "2-4", "5-19", "20+"]


def by_history(frame, at_offset):
    frame = frame.assign(prior=pd.cut(num(frame["dev_prior_launches"]), BINS, labels=NAMES))
    later = explore[(explore["offset_s"] == at_offset) & explore["mint"].isin(frame["mint"])]
    frame = frame.drop(columns=["label_hit_2x"]).merge(
        later[["mint", "label_hit_2x", "graduates_later", "on_curve"]], on="mint", how="left")
    out = {}
    for name, part in frame.groupby("prior", observed=False):
        out[name] = {"coins": len(part), "bad": ci(part, "label_bad"),
                     "graduated (any)": ci(part, "label_graduated_1h"),
                     f"graduates after T+{at_offset} s": ci(part[part["on_curve"]],
                                                             "graduates_later"),
                     f"2x from T+{at_offset} s": ci(part, "label_hit_2x")}
    return out


ordinary = coins[coins["group"] == "ordinary"]
for title, frame in (("All ordinary coins", ordinary),
                     ("Ordinary coins with at least 12 h of backfill before them",
                      ordinary[num(ordinary["history_s"]) >= 12 * 3600])):
    result = by_history(frame, 60)
    print(title)
    display(pd.DataFrame({k: {c: (v[c] if c == "coins" else with_ci(v[c])) for c in v}
                          for k, v in result.items()}).T.rename_axis("earlier launches"))

result = by_history(ordinary, 60)
fig, axes = plt.subplots(1, 2, figsize=(9, 2.8), layout="constrained")
for ax, column in zip(axes, ["bad", "graduates after T+60 s"], strict=True):
    for i, name in enumerate(NAMES):
        r, lo, hi = result[name][column]
        if np.isnan(r):
            continue
        ax.plot([i, i], [lo, hi], color=COLOR["ordinary"])
        ax.plot(i, r, "o", color=COLOR["ordinary"], markersize=7, markeredgecolor=SURFACE,
                markeredgewidth=1.5)
    ax.set_xticks(range(len(NAMES)), NAMES)
    ax.set_xlabel("creator's earlier launches in the data")
    as_pct(ax)
    ax.set_title(f"Ordinary coins: {column}")
plt.show()

## 8. Signals

How the signals are spread, and how well each one alone separates outcomes. This uses
ordinary coins (where every label exists) at T+30 s and T+60 s. Graduation is the
honest version from 2b: coins still on the curve at the cutoff that graduate later.
**AUC** is the chance that
a random coin with the outcome has a higher value than a random coin without it (0.5 =
no information; below 0.5 = lower values go with the outcome). **AP** (average precision)
is compared with the base rate. A single signal with an AUC near 1 would point to a leak.

In [ ]:
SKIP = {"token_age_s", "is_mayhem", "is_sol_paired"}
features = [c for c in manifest["columns"]["features"] if c not in SKIP]
targets = {"bad": "label_bad", "graduates later": "graduates_later",
           "graduates later, coins with 10+ trades": "graduates_later",
           "2x": "label_hit_2x", "2x, coins with 10+ trades": "label_hit_2x"}
results = []
for offset in (30, 60):
    frame = explore[(explore["group"] == "ordinary") & (explore["offset_s"] == offset)]
    for target, column in targets.items():
        part = frame
        if target.startswith("graduates later"):
            part = frame[frame["on_curve"].to_numpy(bool)]
        if target.endswith("10+ trades"):
            part = part[num(part["trades"]) >= 10]  # is it more than "still alive"?
        y_all = num(part[column])
        for feature in features:
            x_all = num(part[feature])
            keep = ~np.isnan(x_all) & ~np.isnan(y_all)
            x, y = x_all[keep], y_all[keep]
            if len(np.unique(y)) < 2 or len(np.unique(x)) < 2:
                continue
            results.append({"cutoff": f"T+{offset} s", "target": target, "signal": feature,
                            "AUC": roc_auc_score(y, x),
                            "AP": average_precision_score(
                                y, x if roc_auc_score(y, x) >= 0.5 else -x),
                            "base rate": y.mean(), "known": keep.mean(), "coins": len(y)})
scores = pd.DataFrame(results)
scores["strength"] = (scores["AUC"] - 0.5).abs() * 2
suspicious = scores[scores["strength"] > 0.9]
print("Signals with an AUC above 0.95 or below 0.05 on their own (check each for a leak):")
display(suspicious[["cutoff", "target", "signal", "AUC", "AP", "base rate"]].round(3)
        if len(suspicious) else "none")
for (cutoff, target), part in scores.groupby(["cutoff", "target"], sort=False):
    top = part.sort_values("strength", ascending=False).head(6)
    print(f"\n{target} at {cutoff} ({top['coins'].iloc[0]:,} coins, base rate "
          f"{top['base rate'].iloc[0]:.1%}):")
    display(top[["signal", "AUC", "AP", "known"]].round(3).set_index("signal"))

In [ ]:
frame = explore[(explore["group"] == "ordinary") & (explore["offset_s"] == 60)
                & explore["on_curve"]]
grad = frame["graduates_later"].to_numpy(bool)
panels = [("unique_buyers", True), ("buy_sol", True), ("curve_progress_pct", False),
          ("top10_hold_pct", False)]
fig, axes = plt.subplots(1, len(panels), figsize=(11, 2.8), layout="constrained")
for ax, (feature, log) in zip(axes, panels, strict=True):
    values = num(frame[feature])
    edges = (np.logspace(np.log10(max(np.nanmin(values[values > 0]), 1e-3)),
                         np.log10(np.nanmax(values)), 30) if log
             else np.linspace(np.nanmin(values), np.nanmax(values), 30))
    for mask, color, label in ((~grad, REST, "did not graduate"),
                               (grad, COLOR["ordinary"], "graduated later")):
        v = values[mask & ~np.isnan(values)]
        if log:
            v = v[v > 0]
        ax.hist(v, bins=edges, weights=np.full(len(v), 1 / max(len(v), 1)), histtype="step",
                color=color, linewidth=1.5, label=label)
    if log:
        ax.set_xscale("log")
    ax.set_title(feature)
    ax.set_yticks([])
    ax.grid(axis="y", visible=False)
axes[0].legend(loc="upper right", fontsize=8)
fig.suptitle("Ordinary coins still on the curve at T+60 s: graduated later in the hour vs "
             "the rest (share of each group per bin; zero values left out on log scales)",
             x=0.01, ha="left", color=INK2, fontsize=9)
plt.show()

quant = {}
for feature, _ in panels + [("holders", False), ("dev_hold_pct", False), ("snipers", False)]:
    values = num(frame[feature])
    for label, mask in (("graduated later", grad), ("did not", ~grad)):
        q = np.nanquantile(values[mask], [0.1, 0.5, 0.9])
        quant[(feature, label)] = {"10%": q[0], "median": q[1], "90%": q[2]}
display(pd.DataFrame(quant).T.round(2))

**Reading a very high AUC.** A signal flagged above is a leak only if it uses events after
the cutoff; R2's no-lookahead tests rule that out for every signal here. What remains is
the question of how much of the outcome is already visible at the cutoff. For graduating
later, curve progress is the main one: a coin that is most of the way up its curve needs
only a few more buys, while most coins sit near 0%. The AP column, compared with the base
rate, says how clean the top of the ranking is. The "10+ trades" rows ask the harder
question among coins that are clearly alive.

## 9. The rules as the first baseline

The hand-written rules in `config/rules.yaml` (the same ones the live engine uses) give
each coin *avoid*, *caution* or *pass*. Here they are scored on the exploration rows,
ordinary coins only (mayhem coins and coins priced in another coin always get a caution
reason, so they never pass). The formal comparison with models comes in R5, on the
held-out days.

- **As a filter for bad coins (stage 1):** flagging *avoid* coins as bad, how many bad
  coins it catches (recall), how many fine coins it wrongly blocks, and how often a
  flagged coin really is bad (precision). The spec's practical target was 90% recall
  while blocking at most 30% of fine coins.
- **As a picker (stages 2-3):** how coins that *pass* turned out against all coins.

In [ ]:
ordinary = explore[explore["group"] == "ordinary"]
shares, filters, pickers = {}, {}, {}
for offset in OFFSETS:
    frame = ordinary[ordinary["offset_s"] == offset]
    verdict = frame["rules_verdict"].astype(str)
    shares[f"T+{offset} s"] = {v: pct((verdict == v).mean()) for v in ("avoid", "caution", "pass")}
    bad = num(frame["label_bad"])
    for flag_name, flag in (("avoid", verdict == "avoid"), ("avoid or caution", verdict != "pass")):
        f = flag.to_numpy().astype(float)
        hours = frame["hour_ts"].to_numpy()
        filters[(f"T+{offset} s", flag_name)] = {
            "recall of bad": with_ci(hour_interval(f[bad == 1], hours[bad == 1], **CI)),
            "fine blocked": with_ci(hour_interval(f[bad == 0], hours[bad == 0], **CI)),
            "precision": with_ci(hour_interval(bad[f == 1], hours[f == 1], **CI)),
        }
    passed = frame[(verdict == "pass").to_numpy()]
    curve, passed_curve = frame[frame["on_curve"]], passed[passed["on_curve"]]
    pickers[f"T+{offset} s"] = {
        "pass coins": len(passed),
        "graduates later: pass": with_ci(ci(passed_curve, "graduates_later")),
        "graduates later: all": with_ci(ci(curve, "graduates_later")),
        "2x: pass": with_ci(ci(passed, "label_hit_2x")),
        "2x: all": with_ci(ci(frame, "label_hit_2x")),
        "bad: pass": with_ci(ci(passed, "label_bad")),
    }
print("Verdicts, ordinary coins:")
display(pd.DataFrame(shares).T)
print("As a filter for bad coins:")
display(pd.DataFrame(filters).T)
print("As a picker:")
display(pd.DataFrame(pickers).T)
base = ci(ordinary[ordinary["offset_s"] == OFFSETS[0]], "label_bad")
print(f"Base rate of bad ordinary coins: {with_ci(base)}. Calling every coin bad catches "
      "all of them and blocks every fine coin.")

## 10. Notes for the models (R5)

- Outcomes are very unbalanced (most coins die; few graduate or double), so PR-AUC and
  precision of the top coins per hour matter more than accuracy.
- Graduation: most "graduated within the hour" coins had already graduated before the
  first decision time. A useful stage 2 target is graduating *later*, among coins still on
  the curve at the decision time (a choice for the spec, R5).
- Several signals carry the same information (curve progress, market cap, net SOL and
  price change all follow the curve's reserves).
- Mayhem coins behave like a different market and are modeled and reported separately.
- `dev_prior_launches` gets fairer with more days of history (R4).
- Coins launched in the same hour move together: keep resampling whole hours, and split
  by day.
- The findings with numbers are in `docs/PROGRESS.md`, "R3".